# Bronze Ingestion: BC Geology (DataBC WFS) — Maple Ridge AOI

The original *BC Digital Surficial Geology* dataset only covers the central Interior Plateau and has **no coverage at Maple Ridge** (coastal Fraser Lowland).

This notebook instead pulls province-wide geology from the **BC Geographic Warehouse (DataBC) WFS**, which fully covers Maple Ridge, clipped to the AOI bounding box:

| Bronze table | DataBC layer | Theme |
|---|---|---|
| `bronze_bc_quaternary_geology` | `GEOL_QUATERNARY_POLY` | Quaternary / surficial geology |
| `bronze_bc_bedrock_geology` | `GEOL_BEDROCK_UNIT_POLY_SVW` | Bedrock geology |
| `bronze_bc_geological_faults` | `GEOL_FAULT_LINE` | Geological faults |

Features are fetched as GeoJSON over plain HTTP (`requests`) — no geopandas, no `%pip install` — and stored raw (geometry + properties as JSON) in the bronze layer.

## 1. Parameters

In [ ]:
# Parameters — Maple Ridge AOI
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20
AOI_NAME = "Maple Ridge, BC"

WFS_ENDPOINT = "https://openmaps.gov.bc.ca/geo/pub/wfs"
MAX_FEATURES = 5000  # safety cap per layer

# DataBC WFS layer -> bronze Delta table (province-wide coverage, includes coastal Maple Ridge)
LAYERS = {
    "bronze_bc_quaternary_geology": "pub:WHSE_MINERAL_TENURE.GEOL_QUATERNARY_POLY",
    "bronze_bc_bedrock_geology":    "pub:WHSE_MINERAL_TENURE.GEOL_BEDROCK_UNIT_POLY_SVW",
    "bronze_bc_geological_faults":  "pub:WHSE_MINERAL_TENURE.GEOL_FAULT_LINE",
}

print("Parameters loaded")
print({
    "aoi": AOI_NAME,
    "lat": LATITUDE,
    "lon": LONGITUDE,
    "radius_km": RADIUS_KM,
    "tables": list(LAYERS.keys()),
})

## 2. Helpers — AOI bbox + WFS fetch

In [ ]:
import math
import json
import requests
from datetime import datetime, timezone


def radius_to_bbox(lat, lon, radius_km):
    # Approximate conversion from km radius to lat/lon degree offsets.
    lat_delta = radius_km / 111.32
    cos_lat = math.cos(math.radians(lat))
    lon_delta = 180.0 if abs(cos_lat) < 1e-8 else radius_km / (111.32 * cos_lat)
    return [lon - lon_delta, lat - lat_delta, lon + lon_delta, lat + lat_delta]


BBOX = radius_to_bbox(LATITUDE, LONGITUDE, RADIUS_KM)
# WFS 2.0 bbox order is miny,minx,maxy,maxx (lat/lon) followed by the CRS urn.
BBOX_PARAM = f"{BBOX[1]},{BBOX[0]},{BBOX[3]},{BBOX[2]},urn:ogc:def:crs:EPSG::4326"
print("AOI bbox [minlon, minlat, maxlon, maxlat]:", BBOX)


def fetch_wfs_features(layer, bbox_param, max_features):
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeNames": layer,
        "count": str(max_features),
        "outputFormat": "application/json",
        "srsName": "EPSG:4326",
        "bbox": bbox_param,
    }
    resp = requests.get(WFS_ENDPOINT, params=params, timeout=180)
    resp.raise_for_status()
    data = resp.json()
    return data.get("features", [])

## 3. Ingestion function — GeoJSON features to bronze Delta

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
)

BRONZE_SCHEMA = StructType([
    StructField("feature_id", StringType(), True),
    StructField("source_layer", StringType(), True),
    StructField("geometry_type", StringType(), True),
    StructField("geometry_json", StringType(), True),
    StructField("properties_json", StringType(), True),
    StructField("aoi_name", StringType(), True),
    StructField("aoi_lat", DoubleType(), True),
    StructField("aoi_lon", DoubleType(), True),
    StructField("aoi_radius_km", DoubleType(), True),
    StructField("ingested_at", TimestampType(), True),
])


def ingest_layer(table_name, layer):
    feats = fetch_wfs_features(layer, BBOX_PARAM, MAX_FEATURES)
    print(f"{table_name}: {len(feats)} features fetched from {layer}")
    now = datetime.now(timezone.utc)
    rows = []
    for f in feats:
        geom = f.get("geometry") or {}
        rows.append((
            str(f.get("id")),
            layer,
            geom.get("type"),
            json.dumps(geom),
            json.dumps(f.get("properties", {})),
            AOI_NAME,
            float(LATITUDE),
            float(LONGITUDE),
            float(RADIUS_KM),
            now,
        ))
    if not rows:
        print(f"WARNING: no features for {table_name}; writing empty table with schema")
    df = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table_name))
    return df.count()

## 4. Run ingestion for all layers

In [ ]:
results = {}
for table_name, layer in LAYERS.items():
    results[table_name] = ingest_layer(table_name, layer)

print("\nIngestion complete:")
for t, c in results.items():
    print(f"  {t}: {c} rows")

## 5. Verify bronze tables

In [ ]:
for table_name in LAYERS:
    cnt = spark.table(table_name).count()
    print(f"{table_name}: {cnt} rows")
    display(spark.table(table_name).limit(3))